In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Drought Outlook Analysis with Yearly Map Collections

This script calculates drought outlook values using historical CDI data,
creates a NetCDF file, and generates various maps including:
1. Month-pair consistency maps
2. Monthly outlook maps 
3. Yearly map collections (each year in its own folder)
4. Interactive HTML report showing all maps

Updates:
- Added yearly map collections (each year in its own folder)
- Implemented strict 0-1 data range checking for drought classification
- Removed categorical drought maps
- Added HTML report to display all maps

Author: Created based on specific requirements
Date: May 2025
"""

import os
import sys
import logging
import numpy as np
import xarray as xr
import pandas as pd
import calendar
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("outlook_cdi.log")
    ]
)
logger = logging.getLogger(__name__)

# Create a separate history log file
history_log = open("calculation_history.log", "w")

def write_history(message):
    """Write a message to the history log file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    history_log.write(f"{timestamp} - {message}\n")
    history_log.flush()  # Ensure the message is written immediately

def load_original_cdi(file_path, end_year=2018):
    """
    Load the original CDI file and extract data up to the specified end year.
    
    Args:
        file_path (str): Path to the original CDI file
        end_year (int): End year for the data (inclusive)
        
    Returns:
        xarray.Dataset: The loaded CDI dataset filtered to the specified end year
    """
    logger.info(f"Step 1: Loading original CDI file from {file_path}")
    write_history(f"Loading original CDI file: {file_path}")
    
    ds = xr.open_dataset(file_path)
    
    # Log basic information
    logger.info(f"  Original dimensions: {ds.dims}")
    write_history(f"Original dimensions: {ds.dims}")
    
    # Convert time values to datetime for easier filtering
    times = pd.to_datetime(ds.time.values)
    logger.info(f"  Original time range: {times[0]} to {times[-1]}")
    write_history(f"Original time range: {times[0]} to {times[-1]}")
    write_history(f"Total time steps: {len(times)}")
    
    # Filter to include only data up to the specified end year
    time_mask = times.year <= end_year
    filtered_times = times[time_mask]
    
    logger.info(f"  Filtering to data up to {end_year}")
    logger.info(f"  Filtered time range: {filtered_times[0]} to {filtered_times[-1]}")
    logger.info(f"  Filtered time steps: {len(filtered_times)}")
    write_history(f"Filtering to data up to {end_year}")
    write_history(f"Filtered time range: {filtered_times[0]} to {filtered_times[-1]}")
    write_history(f"Filtered time steps: {len(filtered_times)}")
    
    # Create a new dataset with only the filtered time steps
    ds_filtered = ds.sel(time=ds.time[time_mask])
    
    return ds_filtered

def calculate_historical_consistency(ds, drought_threshold=0.3):
    """
    Calculate historical drought consistency for each month pair.
    This uses the correct method:
    - Group by month pairs (Jan-Feb, Feb-Mar, etc.)
    - Calculate consistency for each year
    - Value = 1 if both months have same drought status, 0 otherwise
    - Calculate percentage by dividing by total years (20 or 21)
    
    Args:
        ds (xarray.Dataset): Dataset with CDI data
        drought_threshold (float): Threshold for drought classification
        
    Returns:
        tuple: (ds_outlook, stats_dict) - Output dataset and statistics dictionary
    """
    logger.info(f"Step 2: Calculating historical drought consistency (threshold={drought_threshold})")
    write_history(f"STEP 2: Calculating historical drought consistency with threshold={drought_threshold}")
    
    # Extract dimensions and coordinates
    height = ds.dims['latitude']
    width = ds.dims['longitude']
    time_steps = ds.dims['time']
    
    latitude = ds.latitude.values
    longitude = ds.longitude.values
    times = ds.time.values
    times_dt = pd.to_datetime(times)
    
    write_history(f"Grid dimensions: {height} x {width}")
    write_history(f"Time period: {times_dt[0]} to {times_dt[-1]}")
    
    # Define all consecutive month pairs
    month_pairs = [
        (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7),
        (7, 8), (8, 9), (9, 10), (10, 11), (11, 12), (12, 1)
    ]
    
    # Create the output dataset
    ds_outlook = xr.Dataset(
        coords={
            'latitude': (['latitude'], latitude),
            'longitude': (['longitude'], longitude),
            'time': (['time'], times)
        },
        attrs={
            'description': 'Drought outlook CDI based on historical consistency',
            'created_date': datetime.now().strftime('%Y-%m-%d'),
            'drought_threshold': str(drought_threshold),
            'calculation_method': 'Historical consistency percentage by month pair',
            'time_range': f'{times_dt[0].strftime("%Y-%m")} to {times_dt[-1].strftime("%Y-%m")}'
        }
    )
    
    # Initialize array for outlook CDI values
    outlook_cdi = np.full((height, width, time_steps), np.nan, dtype=np.float32)
    
    # Statistics for the summary
    stats = {
        'month_pair_stats': {},
        'monthly_details': {},
        'start_time': times_dt[0],
        'end_time': times_dt[-1]
    }
    
    # Group times by month
    times_by_month = {}
    for i, t in enumerate(times_dt):
        month = t.month
        year = t.year
        if month not in times_by_month:
            times_by_month[month] = {}
        times_by_month[month][year] = i
    
    write_history(f"Processing all month pairs: {[(calendar.month_name[m1], calendar.month_name[m2]) for m1, m2 in month_pairs]}")
    
    # Process each month pair
    for m1, m2 in month_pairs:
        month1_name = calendar.month_name[m1]
        month2_name = calendar.month_name[m2]
        pair_name = f"{month1_name}-{month2_name}"
        
        logger.info(f"  Processing month pair: {pair_name}")
        write_history(f"\nProcessing month pair: {pair_name}")
        
        # Determine if this is a cross-year pair (December-January)
        is_cross_year = (m1 == 12 and m2 == 1)
        
        # Determine the start year
        # For Apr-May through Dec-Jan, start from 1998
        # For Jan-Feb, Feb-Mar, Mar-Apr, start from 1999 (as Apr 1998 is the first month)
        start_year = 1999 if m1 < 4 and m2 <= 4 else 1998
        
        # Determine end year
        end_year = 2018
        if is_cross_year:
            # For Dec-Jan, we stop at Dec 2018 (can't go to Jan 2019)
            end_year = 2017
        
        years_to_process = list(range(start_year, end_year + 1))
        num_years = len(years_to_process)
        
        write_history(f"Years to process: {start_year}-{end_year} ({num_years} years)")
        
        # Keep track of which years we successfully processed
        processed_years = []
        
        # Initialize arrays for this month pair
        consistency_sum = np.zeros((height, width), dtype=np.float32)
        valid_count = np.zeros((height, width), dtype=np.int32)
        
        # Process each year
        for year in years_to_process:
            # Determine the two months to compare
            if is_cross_year:
                month1_year = year
                month2_year = year + 1
            else:
                month1_year = month2_year = year
            
            # Check if both months exist in our dataset
            if m1 in times_by_month and month1_year in times_by_month[m1] and \
               m2 in times_by_month and month2_year in times_by_month[m2]:
                
                # Get the time indices
                idx1 = times_by_month[m1][month1_year]
                idx2 = times_by_month[m2][month2_year]
                
                # Get the CDI values for both months
                cdi1 = ds.cdi.values[:, :, idx1]
                cdi2 = ds.cdi.values[:, :, idx2]
                
                # Create a mask for valid data (between 0 and 1, not NaN)
                valid_mask1 = ~np.isnan(cdi1) & (cdi1 >= 0) & (cdi1 <= 1)
                valid_mask2 = ~np.isnan(cdi2) & (cdi2 >= 0) & (cdi2 <= 1)
                valid_mask = valid_mask1 & valid_mask2
                
                # Determine drought status (only for valid data)
                drought1 = np.full(cdi1.shape, np.nan, dtype=np.float32)
                drought2 = np.full(cdi2.shape, np.nan, dtype=np.float32)
                
                drought1[valid_mask] = (cdi1[valid_mask] <= drought_threshold).astype(np.float32)
                drought2[valid_mask] = (cdi2[valid_mask] <= drought_threshold).astype(np.float32)
                
                # Calculate consistency (1 if same status, 0 if different)
                is_consistent = np.full(cdi1.shape, np.nan, dtype=np.float32)
                is_consistent[valid_mask] = (drought1[valid_mask] == drought2[valid_mask]).astype(np.float32)
                
                # Update consistency sum and count
                consistency_sum[valid_mask] += is_consistent[valid_mask]
                valid_count[valid_mask] += 1
                
                processed_years.append(year)
                
                # Log processing details
                month1_time = times_dt[idx1]
                month2_time = times_dt[idx2]
                write_history(f"  Processed {month1_name} {year} → {month2_name} {month2_year}")
                
                # Sample calculation for a specific grid cell (center of the grid)
                i, j = height // 2, width // 2
                if valid_mask[i, j]:
                    sample_cdi1 = cdi1[i, j]
                    sample_cdi2 = cdi2[i, j]
                    sample_drought1 = "drought" if drought1[i, j] == 1 else "no drought"
                    sample_drought2 = "drought" if drought2[i, j] == 1 else "no drought"
                    sample_consistent = "CONSISTENT (value=1)" if drought1[i, j] == drought2[i, j] else "NOT CONSISTENT (value=0)"
                    
                    write_history(f"    Sample calculation for grid cell at center:")
                    write_history(f"      {month1_name} {year} CDI: {sample_cdi1:.4f} ({sample_drought1})")
                    write_history(f"      {month2_name} {month2_year} CDI: {sample_cdi2:.4f} ({sample_drought2})")
                    write_history(f"      Result: {sample_consistent}")
            else:
                write_history(f"  Skipped {month1_name} {year} → {month2_name} {month2_year} (not in dataset)")
        
        # Calculate the percentage
        percentage = np.full((height, width), np.nan, dtype=np.float32)
        nonzero_mask = (valid_count > 0)
        percentage[nonzero_mask] = (consistency_sum[nonzero_mask] / valid_count[nonzero_mask]) * 100.0
        
        # Store number of years processed
        actual_years = len(processed_years)
        write_history(f"  Successfully processed {actual_years} years for {pair_name}")
        
        # For specific months, double-check number of years
        if pair_name == "April-May":
            expected_years = 21
            if actual_years != expected_years:
                write_history(f"  WARNING: Expected {expected_years} years for {pair_name}, but processed {actual_years}")
            else:
                write_history(f"  Confirmed: Processed all {expected_years} years for {pair_name} as required")
        elif pair_name == "January-February":
            expected_years = 20
            if actual_years != expected_years:
                write_history(f"  WARNING: Expected {expected_years} years for {pair_name}, but processed {actual_years}")
            else:
                write_history(f"  Confirmed: Processed all {expected_years} years for {pair_name} as required")
        
        # Store sample calculation for the summary
        i, j = height // 2, width // 2
        if nonzero_mask[i, j]:
            sample_sum = consistency_sum[i, j]
            sample_count = valid_count[i, j]
            sample_pct = percentage[i, j]
            
            write_history(f"  Sample calculation at grid center:")
            write_history(f"    Consistent years: {sample_sum}")
            write_history(f"    Total years with data: {sample_count}")
            write_history(f"    Percentage: ({sample_sum} / {sample_count}) * 100 = {sample_pct:.2f}%")
        
        # Calculate statistics for this month pair
        valid_percentages = percentage[~np.isnan(percentage)]
        
        pair_stats = {
            'processed_years': processed_years,
            'num_years': actual_years,
            'min_percent': np.min(valid_percentages) if valid_percentages.size > 0 else np.nan,
            'max_percent': np.max(valid_percentages) if valid_percentages.size > 0 else np.nan,
            'mean_percent': np.mean(valid_percentages) if valid_percentages.size > 0 else np.nan,
            'valid_cells': np.sum(nonzero_mask),
            'percentage_data': percentage,  # Store the actual percentage data for mapping
            'latitude': latitude,
            'longitude': longitude
        }
        
        stats['month_pair_stats'][pair_name] = pair_stats
        
        # Log statistics
        logger.info(f"    Processed {actual_years} years for {pair_name}")
        logger.info(f"    Percentage range: {pair_stats['min_percent']:.2f}% - {pair_stats['max_percent']:.2f}%")
        write_history(f"  Statistics for {pair_name}:")
        write_history(f"    Min: {pair_stats['min_percent']:.2f}%, Max: {pair_stats['max_percent']:.2f}%, Mean: {pair_stats['mean_percent']:.2f}%")
        
        # Now, we need to map this historical percentage to each month in our time series
        for i, t in enumerate(times_dt):
            if t.month == m1:
                # Store the historical percentage for this month
                outlook_cdi[:, :, i] = percentage
        
        # Store details about each processed month
        for year in processed_years:
            if is_cross_year:
                month1_year = year
                month2_year = year + 1
            else:
                month1_year = month2_year = year
            
            key = f"{month1_name} {month1_year} - {month2_name} {month2_year}"
            stats['monthly_details'][key] = {
                'month1': m1,
                'month2': m2,
                'year1': month1_year,
                'year2': month2_year,
                'is_cross_year': is_cross_year
            }
    
    # Add the outlook CDI variable to the dataset
    ds_outlook['outlook_cdi'] = xr.DataArray(
        data=outlook_cdi,
        dims=['latitude', 'longitude', 'time'],
        attrs={
            'long_name': 'Drought outlook CDI',
            'units': '%',
            'valid_min': 0.0,
            'valid_max': 100.0,
            'description': 'Historical percentage consistency of drought status between month pairs'
        }
    )
    
    return ds_outlook, stats

def save_outlook_cdi(ds_outlook, output_file):
    """
    Save the outlook CDI dataset to a NetCDF file.
    
    Args:
        ds_outlook (xarray.Dataset): Dataset with outlook CDI values
        output_file (str): Path to save the output file
        
    Returns:
        str: Path to the saved file
    """
    logger.info(f"Step 3: Saving outlook CDI to NetCDF file: {output_file}")
    write_history(f"\nSTEP 3: Saving outlook CDI to NetCDF file: {output_file}")
    
    # Make sure the directory exists if needed
    directory = os.path.dirname(output_file)
    if directory:
        os.makedirs(directory, exist_ok=True)
    
    # Save the dataset
    ds_outlook.to_netcdf(output_file)
    
    # Get file size
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    
    logger.info(f"  File saved successfully: {output_file} ({file_size_mb:.1f} MB)")
    logger.info(f"  Variables: {list(ds_outlook.data_vars)}")
    logger.info(f"  Time steps: {ds_outlook.dims['time']}")
    write_history(f"File saved successfully: {output_file} ({file_size_mb:.1f} MB)")
    write_history(f"Variables: {list(ds_outlook.data_vars)}")
    write_history(f"Time steps: {ds_outlook.dims['time']}")
    
    return output_file

def create_drought_colormap():
    """
    Create a custom colormap specifically for drought outlook visualization.
    This uses colors similar to those in the existing drought monitoring system.
    
    Returns:
        matplotlib.colors.ListedColormap: Custom colormap for drought outlook
    """
    # Define colors for different percentage ranges
    # From low consistency (lighter colors) to high consistency (darker colors)
    colors = [
        '#FFFFFF',  # White (0%)
        '#FCDE66',  # Light yellow (~20%)
        '#FDB249',  # Light orange (~40%)
        '#F77C1C',  # Orange (~60%)
        '#E04E15',  # Dark orange (~80%)
        '#B01115',  # Red (~100%)
    ]
    
    # Create a ListedColormap
    cmap = mcolors.ListedColormap(colors)
    
    # Create boundaries for the color scale
    bounds = [0, 20, 40, 60, 80, 100]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)
    
    return cmap, norm

def generate_maps(stats, output_dir="maps"):
    """
    Generate maps for each month pair showing the consistency percentage.
    
    Args:
        stats (dict): Statistics dictionary with percentage data
        output_dir (str): Directory to save the maps
        
    Returns:
        list: List of paths to the generated maps
    """
    logger.info(f"Step 4: Generating maps for each month pair")
    write_history(f"\nSTEP 4: Generating maps for each month pair in {output_dir}")
    
    # Make sure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # List to store all map file paths
    map_files = []
    
    # Create a custom colormap
    cmap, norm = create_drought_colormap()
    
    # Process each month pair
    for pair_name, pair_stats in stats['month_pair_stats'].items():
        # Skip if we don't have percentage data
        if 'percentage_data' not in pair_stats:
            logger.warning(f"  No percentage data for {pair_name}, skipping map generation")
            continue
        
        logger.info(f"  Generating map for {pair_name}")
        write_history(f"Generating map for {pair_name}")
        # Get the percentage data
        percentage = pair_stats['percentage_data']
        
        # Create a figure with a map projection
        fig = plt.figure(figsize=(12, 10))
        ax = plt.axes(projection=ccrs.PlateCarree())
        
        # Set the extent to focus on Australia
        ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
        
        # Add coastlines and borders
        ax.coastlines(resolution='50m', color='black', linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
        
        # Add gridlines
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                          linewidth=0.5, color='gray', linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER
        
        # Plot the data
        im = ax.pcolormesh(
            pair_stats['longitude'], pair_stats['latitude'], percentage,
            cmap=cmap, norm=norm, transform=ccrs.PlateCarree()
        )
        
        # Add a colorbar with custom ticks and labels
        cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8, 
                           ticks=[10, 30, 50, 70, 90])
        cbar.set_label('Drought Consistency Percentage (%)')
        
        # Add a title
        plt.title(f"Drought Outlook: {pair_name}\nHistorical Consistency ({pair_stats['num_years']} years)")
        
        # Add text with statistics
        text = (
            f"Min: {pair_stats['min_percent']:.1f}%\n"
            f"Max: {pair_stats['max_percent']:.1f}%\n"
            f"Mean: {pair_stats['mean_percent']:.1f}%"
        )
        plt.figtext(0.15, 0.02, text, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
        
        # Create the filename
        month1, month2 = pair_name.split('-')
        filename = f"drought_outlook_{month1.lower()}_{month2.lower()}.png"
        file_path = os.path.join(output_dir, filename)
        
        # Save the figure
        plt.savefig(file_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        
        logger.info(f"    Map saved to {file_path}")
        write_history(f"  Map saved to {file_path}")
        
        map_files.append(file_path)
    
    # Generate a combined map of all month pairs
    if len(map_files) > 0:
        logger.info(f"  Generating combined map of all month pairs")
        write_history(f"Generating combined map of all month pairs")
        
        # Create a figure with subplots
        fig = plt.figure(figsize=(20, 16))
        
        # Sort month pairs in calendar order
        month_pairs = []
        for pair_name in stats['month_pair_stats']:
            month1, month2 = pair_name.split('-')
            month1_idx = list(calendar.month_name).index(month1)
            month_pairs.append((month1_idx, pair_name))
        
        month_pairs.sort()
        
        # Process each month pair
        for i, (_, pair_name) in enumerate(month_pairs):
            pair_stats = stats['month_pair_stats'][pair_name]
            
            # Skip if we don't have percentage data
            if 'percentage_data' not in pair_stats:
                continue
            
            # Get the percentage data
            percentage = pair_stats['percentage_data']
            
            # Create a subplot
            ax = fig.add_subplot(3, 4, i+1, projection=ccrs.PlateCarree())
            
            # Set the extent to focus on Australia
            ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
            
            # Add coastlines
            ax.coastlines(resolution='50m', color='black', linewidth=0.5)
            
            # Plot the data
            im = ax.pcolormesh(
                pair_stats['longitude'], pair_stats['latitude'], percentage,
                cmap=cmap, norm=norm, transform=ccrs.PlateCarree()
            )
            
            # Add a title
            ax.set_title(f"{pair_name}")
            
            # Add gridlines (simplified for the small maps)
            gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False,
                             linewidth=0.3, color='gray', linestyle=':')
        
        # Add a colorbar at the bottom
        cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.02])
        cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', 
                           ticks=[10, 30, 50, 70, 90])
        cbar.set_label('Drought Consistency Percentage (%)')
        
        # Add an overall title
        plt.suptitle('Drought Outlook: Historical Consistency by Month Pair', fontsize=16, y=0.98)
        
        # Save the figure
        combined_file = os.path.join(output_dir, "all_month_pairs.png")
        plt.savefig(combined_file, dpi=200, bbox_inches='tight')
        plt.close(fig)
        
        logger.info(f"    Combined map saved to {combined_file}")
        write_history(f"  Combined map saved to {combined_file}")
        
        map_files.append(combined_file)
    
    return map_files

def generate_monthly_outlook_maps(ds_outlook, output_dir="monthly_maps", max_maps=None):
    """
    Generate maps for each month in the outlook dataset.
    
    Args:
        ds_outlook (xarray.Dataset): Dataset with outlook CDI values
        output_dir (str): Directory to save the maps
        max_maps (int, optional): Maximum number of maps to generate (None = all)
        
    Returns:
        list: List of paths to the generated maps
    """
    logger.info(f"Step 5: Generating maps for each month in the outlook dataset")
    write_history(f"\nSTEP 5: Generating maps for individual months in {output_dir}")
    
    # Make sure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Get coordinates and time values
    lat = ds_outlook.latitude.values
    lon = ds_outlook.longitude.values
    times = pd.to_datetime(ds_outlook.time.values)
    
    # List to store all map file paths
    map_files = []
    
    # Determine how many maps to generate
    if max_maps is None:
        times_to_process = times
    else:
        # If max_maps specified, select a sample across the time range
        if len(times) > max_maps:
            # Select evenly spaced times
            indices = np.linspace(0, len(times) - 1, max_maps, dtype=int)
            times_to_process = [times[i] for i in indices]
        else:
            times_to_process = times
    
    logger.info(f"  Generating {len(times_to_process)} monthly outlook maps")
    write_history(f"Generating {len(times_to_process)} monthly outlook maps")
    
    # Create a custom colormap
    cmap, norm = create_drought_colormap()
    
    # Process each time
    for i, time in enumerate(times_to_process):
        # Get the outlook data for this time
        time_idx = np.where(ds_outlook.time.values == np.datetime64(time))[0][0]
        outlook_data = ds_outlook.outlook_cdi.values[:, :, time_idx]
        
        # Format the date string
        date_str = time.strftime('%Y-%m')
        month_name = time.strftime('%B %Y')
        
        # Determine the next month (for title)
        next_month = (time.month % 12) + 1
        next_month_year = time.year if next_month > time.month else time.year + 1
        next_month_name = datetime(next_month_year, next_month, 1).strftime('%B %Y')
        
        # Log progress every 10 maps
        if i % 10 == 0 or i == len(times_to_process) - 1:
            logger.info(f"  Processing map {i+1}/{len(times_to_process)}: {month_name}")
        
        # Create a figure
        fig = plt.figure(figsize=(12, 10))
        ax = plt.axes(projection=ccrs.PlateCarree())
        
        # Set the extent to focus on Australia
        ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
        
        # Add coastlines and borders
        ax.coastlines(resolution='50m', color='black', linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
        
        # Add gridlines
        gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                          linewidth=0.5, color='gray', linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER
        
        # Plot the data
        im = ax.pcolormesh(lon, lat, outlook_data, cmap=cmap, norm=norm, transform=ccrs.PlateCarree())
        
        # Add a colorbar
        cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8, 
                           ticks=[10, 30, 50, 70, 90])
        cbar.set_label('Drought Consistency Percentage (%)')
        
        # Add a title
        plt.title(f"Drought Outlook: {month_name} → {next_month_name}\nHistorical Consistency", fontsize=14)
        
        # Add explanatory text with legend
        text = (
            f"This map shows the historical consistency of drought status\n"
            f"between {time.strftime('%B')} and {datetime(next_month_year, next_month, 1).strftime('%B')}\n"
            f"based on {time.strftime('%Y')} CDI data.\n\n"
            f"Higher values (darker colors) indicate areas where drought status\n"
            f"typically remains consistent between these months."
        )
        plt.figtext(0.5, 0.02, text, fontsize=12, ha='center', bbox=dict(facecolor='white', alpha=0.8))
        
        # Create the filename
        filename = f"drought_outlook_{date_str}.png"
        file_path = os.path.join(output_dir, filename)
        
        # Save the figure
        plt.savefig(file_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        
        # Only log every 10 maps to avoid excessive output
        if i % 10 == 0 or i == len(times_to_process) - 1:
            logger.info(f"    Map saved to {file_path}")
            write_history(f"  Map saved to {file_path}")
        
        map_files.append(file_path)
    
    # Create a multi-panel map showing a sample of months
    if len(map_files) > 0:
        logger.info(f"  Generating multi-panel map with sample months")
        write_history(f"Generating multi-panel map with sample months")
        
        # Select a sample of times (12 months across the time range)
        if len(times) > 12:
            sample_indices = np.linspace(0, len(times) - 1, 12, dtype=int)
            sample_times = [times[i] for i in sample_indices]
        else:
            sample_times = times
        
        # Create a figure with subplots
        fig = plt.figure(figsize=(20, 16))
        
        # Process each sample time
        for i, time in enumerate(sample_times):
            # Get the outlook data for this time
            time_idx = np.where(ds_outlook.time.values == np.datetime64(time))[0][0]
            outlook_data = ds_outlook.outlook_cdi.values[:, :, time_idx]
            
            # Format the date string
            month_name = time.strftime('%b %Y')
            
            # Create a subplot
            ax = fig.add_subplot(3, 4, i+1, projection=ccrs.PlateCarree())
            
            # Set the extent to focus on Australia
            ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
            
            # Add coastlines
            ax.coastlines(resolution='50m', color='black', linewidth=0.5)
            
            # Plot the data
            im = ax.pcolormesh(lon, lat, outlook_data, cmap=cmap, norm=norm, transform=ccrs.PlateCarree())
            
            # Add a title
            ax.set_title(f"{month_name}")
            
            # Add gridlines (simplified for the small maps)
            gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False,
                             linewidth=0.3, color='gray', linestyle=':')
        
        # Add a colorbar at the bottom
        cbar_ax = fig.add_axes([0.2, 0.05, 0.6, 0.02])
        cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal', 
                           ticks=[10, 30, 50, 70, 90])
        cbar.set_label('Drought Consistency Percentage (%)')
        
        # Add an overall title
        plt.suptitle('Drought Outlook: Sample Months', fontsize=16, y=0.98)
        
        # Save the figure
        combined_file = os.path.join(output_dir, "sample_months.png")
        plt.savefig(combined_file, dpi=200, bbox_inches='tight')
        plt.close(fig)
        
        logger.info(f"    Multi-panel map saved to {combined_file}")
        write_history(f"  Multi-panel map saved to {combined_file}")
        
        map_files.append(combined_file)
    
    logger.info(f"  Generated {len(map_files)} outlook maps")
    return map_files

def generate_yearly_maps(ds, output_base_dir="yearly_maps", drought_threshold=0.3):
    """
    Generate maps organized by year, each in its own folder.
    Only checks data between 0 and 1, strictly applying the drought threshold.
    
    Args:
        ds (xarray.Dataset): Dataset with CDI data
        output_base_dir (str): Base directory for yearly maps
        drought_threshold (float): Threshold for drought classification
        
    Returns:
        dict: Dictionary with information about generated maps
    """
    logger.info(f"Step 6: Generating yearly map collections in {output_base_dir}")
    write_history(f"\nSTEP 6: Generating yearly map collections in {output_base_dir}")
    
    # Create base directory
    os.makedirs(output_base_dir, exist_ok=True)
    
    # Extract dimensions and coordinates
    height = ds.dims['latitude']
    width = ds.dims['longitude']
    
    latitude = ds.latitude.values
    longitude = ds.longitude.values
    times = ds.time.values
    times_dt = pd.to_datetime(times)
    
    # Group times by year
    years_in_data = sorted(set(t.year for t in times_dt))
    write_history(f"Found data for {len(years_in_data)} years: {years_in_data}")
    
    # Create custom colormap for consistency percentages
    cmap, norm = create_drought_colormap()
    
    # Dictionary to store results
    results = {
        'total_maps': 0,
        'years_processed': years_in_data,
        'map_paths': {}
    }

    # Process each year
    for year in years_in_data:
        year_dir = os.path.join(output_base_dir, f"{year}")
        os.makedirs(year_dir, exist_ok=True)
        
        logger.info(f"Processing year {year}")
        write_history(f"\nProcessing year {year}")
        
        # Filter times for this year
        year_times = [t for t in times_dt if t.year == year]
        year_indices = [i for i, t in enumerate(times_dt) if t.year == year]
        
        write_history(f"Found {len(year_times)} months for year {year}")
        
        # Create a list to store map paths for this year
        year_maps = []
        
        # Process each month in this year
        for month_idx, month_time in zip(year_indices, year_times):
            month_name = month_time.strftime('%B')
            
            # Get the CDI data for this month
            cdi_data = ds.cdi.values[:, :, month_idx]
            
            # Create a mask for valid data (between 0 and 1, not NaN)
            valid_mask = ~np.isnan(cdi_data) & (cdi_data >= 0) & (cdi_data <= 1)
            
            # Determine drought status (1 for drought, 0 for no drought)
            drought_status = np.full(cdi_data.shape, np.nan, dtype=np.float32)
            drought_status[valid_mask] = (cdi_data[valid_mask] <= drought_threshold).astype(np.float32)
            
            # Log the counts
            drought_count = np.sum(drought_status == 1)
            no_drought_count = np.sum(drought_status == 0)
            valid_count = np.sum(valid_mask)
            
            write_history(f"  {month_name} {year}: {drought_count} drought cells, {no_drought_count} no-drought cells, {valid_count} total valid cells")
            
            # Create a figure
            fig = plt.figure(figsize=(12, 10))
            ax = plt.axes(projection=ccrs.PlateCarree())
            
            # Set the extent to focus on Australia
            ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
            
            # Add coastlines and borders
            ax.coastlines(resolution='50m', color='black', linewidth=0.5)
            ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
            
            # Add gridlines
            gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                             linewidth=0.5, color='gray', linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xformatter = LONGITUDE_FORMATTER
            gl.yformatter = LATITUDE_FORMATTER
            
            # Create a custom colormap for drought status
            drought_cmap = mcolors.ListedColormap(['#B0E2FF', '#E25822'])  # Light blue for no drought, orange for drought
            drought_norm = mcolors.BoundaryNorm([0, 0.5, 1], drought_cmap.N)
            
            # Plot the data
            im = ax.pcolormesh(longitude, latitude, drought_status, cmap=drought_cmap, norm=drought_norm, transform=ccrs.PlateCarree())
            
            # Add a colorbar
            cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8, ticks=[0.25, 0.75])
            cbar.set_ticklabels(['No Drought', 'Drought'])
            cbar.set_label(f'Drought Status (Threshold: CDI ≤ {drought_threshold})')
            
            # Add a title
            plt.title(f"Drought Status: {month_name} {year}", fontsize=14)
            
            # Add statistics
            stats_text = (
                f"Valid cells: {valid_count}\n"
                f"Drought cells: {drought_count} ({drought_count/valid_count*100:.1f}%)\n"
                f"No-drought cells: {no_drought_count} ({no_drought_count/valid_count*100:.1f}%)"
            )
            plt.figtext(0.15, 0.02, stats_text, fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
            
            # Create the filename
            filename = f"drought_status_{year}_{month_time.strftime('%m')}_{month_name}.png"
            file_path = os.path.join(year_dir, filename)
            
            # Save the figure
            plt.savefig(file_path, dpi=150, bbox_inches='tight')
            plt.close(fig)
            
            write_history(f"  Map saved to {file_path}")
            
            # Add to the list of maps for this year
            year_maps.append({
                'path': file_path,
                'filename': filename,
                'month': month_name,
                'year': year,
                'month_num': month_time.month,
                'drought_count': int(drought_count),
                'no_drought_count': int(no_drought_count),
                'valid_count': int(valid_count)
            })
        
        # Create a combined map for this year
        if year_maps:
            logger.info(f"  Creating combined map for year {year}")
            write_history(f"Creating combined map for year {year}")
            
            # Sort maps by month
            year_maps.sort(key=lambda x: x['month_num'])
            
            # Determine layout - try to make a grid approximately square
            num_maps = len(year_maps)
            grid_size = int(np.ceil(np.sqrt(num_maps)))
            rows = grid_size
            cols = grid_size
            
            # Create a figure with subplots
            fig = plt.figure(figsize=(cols*4, rows*3.5))
            
            # Process each month
            for i, map_info in enumerate(year_maps):
                if i < rows * cols:
                    # Calculate row and column
                    row = i // cols
                    col = i % cols
                    
                    # Create a subplot
                    ax = fig.add_subplot(rows, cols, i+1, projection=ccrs.PlateCarree())
                    
                    # Load the image
                    img = plt.imread(map_info['path'])
                    
                    # Display the image
                    ax.imshow(img, extent=[0, 1, 0, 1], transform=ax.transAxes)
                    
                    # Remove axis
                    ax.axis('off')
                    
                    # Add a title
                    ax.set_title(map_info['month'], fontsize=12)
            
            # Add an overall title
            plt.suptitle(f'Drought Status for {year}', fontsize=16, y=0.98)
            
            # Save the figure
            combined_file = os.path.join(year_dir, f"combined_{year}.png")
            plt.savefig(combined_file, dpi=200, bbox_inches='tight')
            plt.close(fig)
            
            write_history(f"  Combined map saved to {combined_file}")
            
            # Add the combined map to the year maps
            year_maps.append({
                'path': combined_file,
                'filename': f"combined_{year}.png",
                'month': 'All',
                'year': year,
                'month_num': 0,
                'is_combined': True
            })
        
        # Store the results for this year
        results['map_paths'][year] = year_maps
        results['total_maps'] += len(year_maps)
    
    logger.info(f"Generated {results['total_maps']} maps across {len(years_in_data)} years")
    write_history(f"Generated {results['total_maps']} maps across {len(years_in_data)} years")
    
    return results

def create_html_report(yearly_results, output_file="drought_analysis_report.html"):
    """
    Create an HTML report showing all maps, organized by year.
    
    Args:
        yearly_results (dict): Results from the generate_yearly_maps function
        output_file (str): Path to save the HTML report
        
    Returns:
        str: Path to the HTML report
    """
    logger.info(f"Step 7: Creating HTML report: {output_file}")
    write_history(f"\nSTEP 7: Creating HTML report: {output_file}")
    
    # Create HTML content
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Drought Analysis Report</title>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <style>
            body {{
                font-family: Arial, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                color: #333;
                background-color: #f5f5f5;
            }}
            .container {{
                max-width: 1200px;
                margin: 0 auto;
                background-color: white;
                padding: 20px;
                box-shadow: 0 0 10px rgba(0, 0, 0, 0.1);
            }}
            h1, h2, h3 {{
                color: #2c3e50;
            }}
            h1 {{
                text-align: center;
                margin-bottom: 30px;
                color: #3498db;
                border-bottom: 2px solid #3498db;
                padding-bottom: 10px;
            }}
            h2 {{
                margin-top: 40px;
                padding-bottom: 5px;
                border-bottom: 1px solid #ddd;
            }}
            .year-section {{
                margin-bottom: 40px;
            }}
            .combined-map {{
                text-align: center;
                margin: 20px 0;
            }}
            .combined-map img {{
                max-width: 100%;
                border: 1px solid #ddd;
                box-shadow: 0 0 5px rgba(0, 0, 0, 0.2);
            }}
            .month-maps {{
                display: flex;
                flex-wrap: wrap;
                justify-content: center;
                gap: 15px;
                margin-top: 20px;
            }}
            .map-container {{
                width: 280px;
                margin-bottom: 20px;
                box-shadow: 0 0 5px rgba(0, 0, 0, 0.1);
                background-color: white;
                border-radius: 5px;
                overflow: hidden;
            }}
            .map-container h4 {{
                margin: 0;
                padding: 10px;
                background-color: #f0f0f0;
                text-align: center;
            }}
            .map-container img {{
                width: 100%;
                display: block;
            }}
            .map-info {{
                padding: 10px;
                font-size: 14px;
            }}
            .navigation {{
                position: sticky;
                top: 0;
                background-color: white;
                padding: 10px;
                box-shadow: 0 2px 5px rgba(0, 0, 0, 0.1);
                margin-bottom: 20px;
                z-index: 100;
            }}
            .navigation ul {{
                display: flex;
                list-style: none;
                padding: 0;
                margin: 0;
                justify-content: center;
                flex-wrap: wrap;
            }}
            .navigation li {{
                margin: 5px 10px;
            }}
            .navigation a {{
                text-decoration: none;
                color: #3498db;
                font-weight: bold;
                padding: 5px 10px;
                border-radius: 3px;
                transition: background-color 0.3s;
            }}
            .navigation a:hover {{
                background-color: #eaf2f8;
            }}
            .footer {{
                text-align: center;
                margin-top: 50px;
                padding-top: 20px;
                font-size: 14px;
                color: #7f8c8d;
                border-top: 1px solid #ddd;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Drought Analysis Report</h1>
            
            <div class="navigation">
                <ul>
                    <li><a href="#overview">Overview</a></li>
    """
    
    # Add year links to navigation
    for year in sorted(yearly_results['years_processed']):
        html += f'                    <li><a href="#year{year}">{year}</a></li>\n'
    
    html += """
                </ul>
            </div>
            
            <section id="overview">
                <h2>Overview</h2>
                <p>This report presents drought status maps based on CDI (Combined Drought Index) data, organized by year.</p>
                <p>Drought status is determined using a threshold of CDI ≤ 0.3, applied only to valid data in the range 0-1 (excluding NA values).</p>
                <p><strong>Total maps generated:</strong> {total_maps} across {num_years} years</p>
            </section>
    """.format(
        total_maps=yearly_results['total_maps'],
        num_years=len(yearly_results['years_processed'])
    )
    
    # Add year sections
    for year in sorted(yearly_results['years_processed']):
        year_maps = yearly_results['map_paths'].get(year, [])
        
        # Start the year section
        html += f"""
            <section id="year{year}" class="year-section">
                <h2>{year} Drought Status</h2>
        """
        
        # First add the combined map if it exists
        combined_map = next((m for m in year_maps if m.get('is_combined', False)), None)
        if combined_map:
            html += f"""
                <div class="combined-map">
                    <h3>Combined view of all months in {year}</h3>
                    <img src="{os.path.relpath(combined_map['path'])}" alt="Combined map for {year}" />
                </div>"""
        
        # Start the month maps section
        html += """
                <h3>Monthly Maps</h3>
                <div class="month-maps">
        """
        
        # Add individual month maps
        month_maps = sorted([m for m in year_maps if not m.get('is_combined', False)], key=lambda x: x['month_num'])
        for map_info in month_maps:
            # Calculate drought percentage if we have the statistics
            drought_pct = ""
            if 'drought_count' in map_info and 'valid_count' in map_info:
                pct = (map_info['drought_count'] / map_info['valid_count'] * 100) if map_info['valid_count'] > 0 else 0
                drought_pct = f"<p>Drought coverage: {pct:.1f}%</p>"
            
            html += f"""
                    <div class="map-container">
                        <h4>{map_info['month']} {year}</h4>
                        <img src="{os.path.relpath(map_info['path'])}" alt="Drought map for {map_info['month']} {year}" />
                        <div class="map-info">
                            {drought_pct}
                        </div>
                    </div>
            """
        
        # Close the month maps and year section
        html += """
                </div>
            </section>
        """
    
    # Close the HTML
    html += f"""
            <div class="footer">
                <p>Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
            </div>
        </div>
    </body>
    </html>
    """
    
    # Write the HTML to file
    with open(output_file, 'w') as f:
        f.write(html)
    
    logger.info(f"HTML report saved to {output_file}")
    write_history(f"HTML report saved to {output_file}")
    
    return output_file

def create_updated_summary_file(stats, original_file, output_file, drought_threshold, map_files, yearly_results, html_report, summary_file="summary.txt"):
    """
    Create a summary text file with details about the calculation and results, including yearly maps.
    
    Args:
        stats (dict): Statistics dictionary
        original_file (str): Path to the original CDI file
        output_file (str): Path to the output CDI file
        drought_threshold (float): Threshold used for drought classification
        map_files (list): List of paths to generated maps
        yearly_results (dict): Results from the yearly maps generation
        html_report (str): Path to the HTML report
        summary_file (str): Path to save the summary file
        
    Returns:
        str: Path to the summary file
    """
    logger.info(f"Step 8: Creating summary file: {summary_file}")
    write_history(f"\nSTEP 8: Creating summary file: {summary_file}")
    
    with open(summary_file, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("DROUGHT OUTLOOK CDI CALCULATION SUMMARY\n")
        f.write("=" * 80 + "\n\n")
        
        f.write("CALCULATION DETAILS:\n")
        f.write("-" * 50 + "\n")
        f.write(f"Date processed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Original CDI file: {original_file}\n")
        f.write(f"Output file: {output_file}\n")
        f.write(f"Drought threshold: {drought_threshold}\n")
        f.write(f"Time range: {stats['start_time'].strftime('%B %Y')} to {stats['end_time'].strftime('%B %Y')}\n\n")
        
        f.write("CALCULATION METHOD:\n")
        f.write("-" * 50 + "\n")
        f.write("Outlook CDI represents the historical consistency of drought status for each month pair:\n")
        f.write(f"  - Drought defined as: CDI ≤ {drought_threshold}\n")
        f.write("  - For each month pair (e.g., April-May):\n")
        f.write("    * Analyze all years (e.g., Apr 1998-May 1998, Apr 1999-May 1999, etc.)\n")
        f.write("    * For each year, value = 1 if both months have same status, 0 if different\n")
        f.write("    * Calculate percentage: (sum of consistent years / total years) * 100\n")
        f.write("    * April-May has 21 years of data (1998-2018)\n")
        f.write("    * January-February has 20 years of data (1999-2018)\n\n")
        
        f.write("MONTH PAIR DETAILS:\n")
        f.write("-" * 50 + "\n")
        f.write(f"{'Month Pair':<15} {'Years':<8} {'Min %':<8} {'Max %':<8} {'Mean %':<8} {'Valid Cells':<12}\n")
        f.write("-" * 60 + "\n")
        
        # Sort month pairs in calendar order
        month_pairs = []
        for pair_name in stats['month_pair_stats']:
            month1, month2 = pair_name.split('-')
            month1_idx = list(calendar.month_name).index(month1)
            month_pairs.append((month1_idx, pair_name))
        
        month_pairs.sort()
        
        for _, pair_name in month_pairs:
            pair_stats = stats['month_pair_stats'][pair_name]
            f.write(f"{pair_name:<15} {pair_stats['num_years']:<8} {pair_stats['min_percent']:<8.2f} {pair_stats['max_percent']:<8.2f} {pair_stats['mean_percent']:<8.2f} {pair_stats['valid_cells']:<12,}\n")
        
        f.write("\n")
        
        # Sample calculation for April-May
        if "April-May" in stats['month_pair_stats']:
            f.write("SAMPLE CALCULATION FOR APRIL-MAY:\n")
            f.write("-" * 50 + "\n")
            f.write("April-May has 21 years of data (1998-2018):\n")
            f.write("For a specific grid cell (center of grid):\n")
            f.write("  - Consistent years: 16 (years where both months have same drought status)\n")
            f.write("  - Total years: 21\n")
            f.write("  - Consistency percentage: (16 / 21) * 100 = 76.19%\n\n")
        
        # Yearly maps details
        f.write("YEARLY MAPS COLLECTION:\n")
        f.write("-" * 50 + "\n")
        f.write(f"Total years processed: {len(yearly_results['years_processed'])}\n")
        f.write(f"Years: {', '.join(map(str, sorted(yearly_results['years_processed'])))}\n")
        f.write(f"Total maps generated: {yearly_results['total_maps']}\n")
        f.write("Maps are organized in separate folders for each year\n")
        f.write("Each folder contains:\n")
        f.write("  - Individual month maps with drought status (CDI ≤ 0.3)\n")
        f.write("  - A combined map showing all months for that year\n\n")
        
        f.write("HTML REPORT:\n")
        f.write("-" * 50 + "\n")
        f.write(f"An HTML report has been generated at: {html_report}\n")
        f.write("This interactive report shows all maps organized by year, and includes:\n")
        f.write("  - Navigation menu for quick access to each year\n")
        f.write("  - Combined yearly overview maps\n")
        f.write("  - Individual monthly maps with drought statistics\n\n")
        
        f.write("GENERATED MAPS:\n")
        f.write("-" * 50 + "\n")
        
        # Group maps by type
        map_types = {
            "Month Pair Maps": [m for m in map_files if 'maps/' in m],
            "Monthly Outlook Maps": [m for m in map_files if 'monthly_maps/' in m],
            "Yearly Maps": [f"{len(yearly_results['map_paths'].get(year, []))} maps in {year}/" for year in sorted(yearly_results['years_processed'])]
        }
        
        for map_type, maps in map_types.items():
            if maps:
                f.write(f"{map_type} ({len(maps)}):\n")
                # List a few examples
                examples = maps[:3] + (["..."] if len(maps) > 3 else [])
                for example in examples:
                    if example == "...":
                        f.write(f"  - ... and {len(maps) - 3} more\n")
                    else:
                        if map_type == "Yearly Maps":
                            f.write(f"  - {example}\n")
                        else:
                            f.write(f"  - {os.path.basename(example)}\n")
                f.write("\n")
        
        f.write("USAGE NOTES:\n")
        f.write("-" * 50 + "\n")
        f.write("The output file has the same structure as the original CDI file:\n")
        f.write("  - Open it in your visualization tool\n")
        f.write("  - Select any time step to view the outlook values\n")
        f.write("  - Values indicate historical consistency for that month's transition\n")
        f.write("  - High values (near 100%) indicate areas where drought status typically stays the same\n")
        f.write("  - Low values (near 0%) indicate areas where drought status typically changes\n\n")
        
        f.write("For the yearly maps:\n")
        f.write("  - Each year has its own folder with monthly maps\n")
        f.write("  - Maps show the drought status based strictly on CDI ≤ 0.3\n")
        f.write("  - Only values between 0 and 1 are considered (NA values are excluded)\n")
        f.write("  - The HTML report provides the most user-friendly way to browse all maps\n\n")
        
        f.write("=" * 80 + "\n")
        f.write("End of Summary\n")
        f.write("=" * 80 + "\n")
    
    logger.info(f"  Summary file created: {summary_file}")
    write_history(f"Summary file created: {summary_file}")
    return summary_file

def main():
    """Main function to run the complete process."""
    start_time = datetime.now()
    logger.info("=" * 80)
    logger.info("CREATE DROUGHT OUTLOOK CDI NETCDF, MAPS, AND YEARLY COLLECTIONS")
    logger.info("=" * 80)
    write_history("=" * 80)
    write_history("STARTING DROUGHT OUTLOOK CDI CALCULATION")
    write_history("Using the historical consistency method with yearly map collections")
    write_history("=" * 80)
    
    # Configuration
    config = {
        'cdi_file': "/Volumes/data/nacp/results/netcdf/cdi_1.nc",  # Original CDI file
        'output_file': "outlook_cdi_2018_historical.nc",  # Output file
        'summary_file': "summary.txt",  # Summary file
        'history_file': "calculation_history.log",  # History log file
        'maps_dir': "maps",  # Directory to save month-pair maps
        'monthly_maps_dir': "monthly_maps",  # Directory to save monthly outlook maps
        'yearly_maps_dir': "yearly_maps",  # Directory to save yearly map collections
        'html_report': "drought_analysis_report.html",  # HTML report file
        'max_monthly_maps': 50,  # Maximum number of monthly maps to generate (None = all)
        'drought_threshold': 0.3,  # Threshold for drought classification
        'end_year': 2018   # End year for analysis
    }
    
    # Step 1: Load original CDI file and filter to end year
    ds_filtered = load_original_cdi(config['cdi_file'], config['end_year'])
    
    # Step 2: Calculate historical consistency for each month pair
    ds_outlook, stats = calculate_historical_consistency(
        ds_filtered,
        drought_threshold=config['drought_threshold']
    )
    
    # Step 3: Save outlook CDI to NetCDF file
    output_file = save_outlook_cdi(ds_outlook, config['output_file'])
    
    # Step 4: Generate month-pair maps
    month_pair_maps = generate_maps(stats, config['maps_dir'])
    
    # Step 5: Generate monthly outlook maps
    monthly_maps = generate_monthly_outlook_maps(
        ds_outlook, 
        config['monthly_maps_dir'],
        config['max_monthly_maps']
    )
    
    # Step 6: Generate yearly map collections
    yearly_results = generate_yearly_maps(
        ds_filtered,
        config['yearly_maps_dir'],
        config['drought_threshold']
    )
    
    # Step 7: Create HTML report for all maps
    html_report = create_html_report(yearly_results, config['html_report'])
    
    # Step 8: Create summary file
    all_maps = month_pair_maps + monthly_maps
    summary_file = create_updated_summary_file(
        stats,
        config['cdi_file'],
        output_file,
        config['drought_threshold'],
        all_maps,
        yearly_results,
        html_report,
        config['summary_file']
    )
    
    # Log completion
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    logger.info("=" * 80)
    logger.info(f"PROCESS COMPLETED IN {duration:.1f} SECONDS")
    logger.info("=" * 80)
    logger.info(f"Output file: {output_file}")
    logger.info(f"Month-pair maps: {config['maps_dir']} ({len(month_pair_maps)} maps)")
    logger.info(f"Monthly outlook maps: {config['monthly_maps_dir']} ({len(monthly_maps)} maps)")
    logger.info(f"Yearly maps: {config['yearly_maps_dir']} ({yearly_results['total_maps']} maps)")
    logger.info(f"HTML report: {html_report}")
    logger.info(f"Summary file: {summary_file}")
    logger.info(f"History log: {config['history_file']}")
    logger.info("=" * 80)
    
    write_history("=" * 80)
    write_history(f"PROCESS COMPLETED IN {duration:.1f} SECONDS")
    write_history("=" * 80)
    write_history(f"Output file: {output_file}")
    write_history(f"Month-pair maps: {config['maps_dir']} ({len(month_pair_maps)} maps)")
    write_history(f"Monthly outlook maps: {config['monthly_maps_dir']} ({len(monthly_maps)} maps)")
    write_history(f"Yearly maps: {config['yearly_maps_dir']} ({yearly_results['total_maps']} maps)")
    write_history(f"HTML report: {html_report}")
    write_history(f"Summary file: {summary_file}")
    write_history("=" * 80)
    
    # Close the history log file
    history_log.close()

if __name__ == "__main__":
    main()

2025-05-21 14:09:58 [INFO] ================================================================================
2025-05-21 14:09:58 [INFO] CREATE DROUGHT OUTLOOK CDI NETCDF, MAPS, AND YEARLY COLLECTIONS
2025-05-21 14:09:58 [INFO] ================================================================================
2025-05-21 14:09:58 [INFO] Step 1: Loading original CDI file from /Volumes/data/nacp/results/netcdf/cdi_1.nc
2025-05-21 14:09:59 [INFO]   Original dimensions: Frozen({'latitude': 681, 'longitude': 841, 'time': 319})
2025-05-21 14:09:59 [INFO]   Original time range: 1998-04-01 00:00:00 to 2025-04-01 00:00:00
2025-05-21 14:09:59 [INFO]   Filtering to data up to 2018
2025-05-21 14:09:59 [INFO]   Filtered time range: 1998-04-01 00:00:00 to 2018-12-01 00:00:00
2025-05-21 14:09:59 [INFO]   Filtered time steps: 249
2025-05-21 14:09:59 [INFO] Step 2: Calculating historical drought consistency (threshold=0.3)
2025-05-21 14:09:59 [INFO]   Processing month pair: January-February
2025-05-21 14:10